# 01 — FB15k-237 Dataset Exploration

Statistics, triple distributions, relation frequencies, and degree analysis.

**No API key required.**

In [1]:
import sys; sys.path.insert(0, '..')
import collections, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from src.data.fb15k237 import FB15k237Dataset
from src.config import Settings

settings = Settings()
print(f'Data dir: {settings.data_dir}')
print(f'Seed    : {settings.random_seed}')

/Users/codex/Code/LLM-Reranked-Knowledge-Graph-Link-Prediction-with-RL-based-Prompt-Policy-Optimization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data dir: data_fb15k237
Seed    : 42


## 1. Load Dataset

In [7]:
dataset = FB15k237Dataset(
	settings.data_dir / 'train.txt',
	settings.data_dir / 'valid.txt',
	settings.data_dir / 'test.txt'
)
train, valid, test = dataset.train, dataset.valid, dataset.test

## 2. Entity & Relation Counts

In [ ]:
all_triples = train + valid + test
entities  = {h for h,r,t in all_triples} | {t for h,r,t in all_triples}
relations = {r for h,r,t in all_triples}
print(f'Entities : {len(entities):,}')
print(f'Relations: {len(relations):,}')

TypeError: unsupported operand type(s) for +: 'PosixPath' and 'PosixPath'

## 3. Split Summary

In [ ]:
print(f'{"Split":<8}{"Triples":>10}{"Entities":>10}{"Relations":>10}')
print('-'*40)
for name, triples in [('train',train),('valid',valid),('test',test)]:
    e={h for h,r,t in triples}|{t for h,r,t in triples}
    r={r for h,r,t in triples}
    print(f'{name:<8}{len(triples):>10,}{len(e):>10,}{len(r):>10,}')

## 4. Top-20 Most Frequent Relations

In [ ]:
rel_c = collections.Counter(r for h,r,t in train)
top   = rel_c.most_common(20)
lbls  = [l.split('/')[-1] for l,_ in top]
cnts  = [c for _,c in top]
fig,ax = plt.subplots(figsize=(12,5))
ax.barh(lbls[::-1], cnts[::-1], color='steelblue')
ax.set_xlabel('Triple count (train)')
ax.set_title('Top-20 Relations in FB15k-237')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_:f'{int(x):,}'))
plt.tight_layout(); plt.show()

## 5. Entity Degree Distribution

In [ ]:
out_deg = list(collections.Counter(h for h,r,t in train).values())
in_deg  = list(collections.Counter(t for h,r,t in train).values())
fig,axes = plt.subplots(1,2,figsize=(13,4))
axes[0].hist(out_deg,bins=60,log=True,color='steelblue',edgecolor='white')
axes[0].set(title='Out-degree (log)',xlabel='Degree',ylabel='Count')
axes[1].hist(in_deg, bins=60,log=True,color='coral',   edgecolor='white')
axes[1].set(title='In-degree (log)', xlabel='Degree',ylabel='Count')
plt.tight_layout(); plt.show()
print(f'Out — mean:{np.mean(out_deg):.1f}  max:{max(out_deg)}')
print(f'In  — mean:{np.mean(in_deg):.1f}   max:{max(in_deg)}')

## 6. Sample Triples

In [ ]:
random.seed(settings.random_seed)
print('10 random train triples:')
for h,r,t in random.sample(train, 10):
    print(f'  {h}  --[{r.split("/")[-1]}]-->  {t}')

## 7. Relation Cardinality

In [ ]:
from collections import defaultdict
rh=defaultdict(set); rt=defaultdict(set)
for h,r,t in train: rh[r].add(h); rt[r].add(t)
card={'1-1':0,'1-N':0,'N-1':0,'N-N':0}
for r in relations:
    th=len(rt[r])/max(len(rh[r]),1); ht=len(rh[r])/max(len(rt[r]),1)
    if th<=1.5 and ht<=1.5: card['1-1']+=1
    elif th>1.5 and ht<=1.5: card['1-N']+=1
    elif th<=1.5 and ht>1.5: card['N-1']+=1
    else: card['N-N']+=1
fig,ax=plt.subplots(figsize=(6,4))
ax.bar(card.keys(),card.values(),color=['#4e79a7','#f28e2b','#59a14f','#e15759'])
ax.set(title='Relation Cardinality Types',ylabel='# relations')
plt.tight_layout(); plt.show()
for k,v in card.items(): print(f'  {k}: {v}')